# 根据测试结果，结合样本之间的距离，得到比较结果

In [54]:
import time,json,os
import numpy as np
import pandas as pd
from JSample import CJSample
from JDistance import CJDistance
from common import *
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from tqdm.notebook import tqdm
from tqdm.notebook import tqdm_notebook
from warnings import filterwarnings
import matplotlib.pyplot as plt
filterwarnings("ignore") 
np.set_printoptions(suppress=True)
pd.set_option('display.float_format',lambda x : '%.8f' % x)
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

# 定义类，用来生成报表

1.根据预测结果，计算每个模型，在不同测试集上的模型评价指标（accuracy，precision，recall和f1_score）。

2.为每个测试集，找到最相似的训练集，根据找到的训练集，找到FDSF的预测结果

In [80]:
class CJTAnalyse:

    def __init__(self):
        self.m_raw_result = pd.read_csv("%sresult.csv"%g_predict_path,index_col=0)
        self.m_raw_similarity = pd.read_csv("%sdistance.csv"%g_predict_path,dtype={'test':object,"train":object},index_col=0)
        self.m_result = pd.DataFrame()
        
    def CalcResult(self):
        df = self.m_raw_result
        all_result = []
        bar = tqdm(total=df.shape[0]-1)
        for model,train_set,data_set,yt,yp in zip(df['model'],df['train'],df['test'],df['y_true'],df['y_pred']):
            y_true = json.loads(yt)
            y_pred = json.loads(yp)
            tmp = {}
            tmp['model'] = model
            if  not train_set in ['baseline','reserve']:
                tmp['train'] = "%02d"%(int(train_set))
            else:
                tmp['train'] = train_set
            if not data_set in ['baseline','reserve']:
                tmp['test'] = "%02d"%(int(data_set))
            else:
                tmp['test'] = data_set
            tmp['acccuracy'] = accuracy_score(y_true,y_pred)
            tmp['precision'] = precision_score(y_true,y_pred)
            tmp['recall'] = recall_score(y_true,y_pred)
            tmp['f1_score'] = f1_score(y_true,y_pred)
            all_result.append(tmp)
            bar.update(1)
        self.m_result = pd.DataFrame(all_result)
        return self.m_result
    
    def SaveResult(self):
        df = self.m_result
        df.to_csv("%sanalyse.csv"%g_predict_path)
    
    def LoadResult(self):
        self.m_result = pd.read_csv("%sanalyse.csv"%g_predict_path,dtype={'test':object,"train":object},index_col=0)
        return self.m_result
    
    def FindTrainSample(self,test_name):
        df = self.m_raw_similarity.copy(deep = True )
        mask = (df['test']==test_name)&(df['train']!="baseline")
        tmp = df[mask].sort_values(by=['Cosine']).tail(3)
        tmp = tmp[mask].sort_values(by=['EMD'])
        tmp = tmp.head( 1 )
        return tmp
    
    def FindBest(self,model,test_name):
        df_tmp = self.m_result[self.m_result['model'] == model].copy(deep = True )
        mask = ( df_tmp['train'] == 'baseline' ) & ( df_tmp['test'] == test_name )
        df_baseline = df_tmp[mask]
        sample = self.FindTrainSample(test_name)
        mask1 = pd.Series()
        for train_sample in sample['train']:
            if mask1.any():
                mask1 = ( mask1 ) | ( df_tmp['train'] == train_sample )
            else:
                mask1 = ( df_tmp['train'] == train_sample )
        mask1 = (mask1) & ( df_tmp['test'] == test_name)
        df_best = df_tmp[mask1]
        return sample,df_baseline.reset_index(drop=True), df_best.reset_index(drop=True)

# 根据预测结果，生成模型评价指标

# 与baseline比较，生成报表

In [81]:
analyse = CJTAnalyse()
analyse.LoadResult()

,model,train,test,acccuracy,precision,recall,f1_score
0,glm,03,03,0.92860094,0.95257384,0.92694784,0.93958614
1,glm,03,00,0.92755291,0.99168308,0.92731960,0.95842196
2,glm,03,02,0.92967919,0.96918814,0.92942552,0.94889046
3,glm,03,05,0.92972911,0.90329936,0.92900823,0.91597343
4,glm,03,08,0.92913488,0.61934249,0.92640620,0.74237517
...,...,...,...,...,...,...,...
535,rf,01,08,0.98745290,0.97405405,0.91040754,0.94115599
536,rf,01,07,0.97612022,0.98889621,0.89866598,0.94162448
537,rf,01,01,0.91975437,0.99913482,0.90031834,0.94715618
538,rf,01,04,0.94742732,0.99616032,0.90082709,0.94609822


In [82]:
df_report = pd.DataFrame()
df_baseline = pd.DataFrame()
df_best = pd.DataFrame()
for model in analyse.m_result['model'].unique():
    #if model == 'cnn':
    #    continue
    for i in range(9):
        sample,baseline,best = analyse.FindBest(model,"%02d"%i)
        df_baseline = pd.concat([df_baseline,baseline],ignore_index = True)
        df_best = pd.concat([df_best,best],ignore_index = True)

def set_kind(train_set):
    if train_set == 'baseline':
        return 'baseline'
    else:
        return "HDFS"
df_report = pd.concat([df_baseline,df_best],ignore_index = True)
df_report['kind'] = df_report.apply(lambda x:set_kind(x['train']),axis=1)
df_report.rename(columns={"acccuracy":"accuracy"},inplace=True)

In [83]:
df_compare = df_report.groupby(['kind','model']).mean().reset_index()
df_best = df_compare[df_compare['kind']=="HDFS"].sort_values(by='model',ascending=False).reset_index(drop=True)
df_base = df_compare[df_compare['kind']!="HDFS"].sort_values(by='model',ascending=False).reset_index(drop=True)
df_diff = pd.DataFrame()
df_diff['model'] = df_best['model']
for measure in ['accuracy','precision','recall','f1_score']:
    df_diff[measure] = df_best[measure] - df_base[measure]
del df_best['kind']
del df_base['kind']
display(df_base,df_best,df_diff,df_diff.mean())

,model,accuracy,precision,recall,f1_score
0,rf,0.95278543,0.98372349,0.91449470,0.94775287
1,glm,0.84405600,0.91078779,0.73230034,0.80963842
2,gbm,0.93847576,0.97823459,0.88887478,0.93125720
3,deeplearn,0.93547946,0.98344919,0.88053067,0.92903295
4,cnn,0.48491836,0.09010857,0.00077841,0.00152200
5,bayes,0.87497562,0.94163116,0.77915949,0.85167756


,model,accuracy,precision,recall,f1_score
0,rf,0.95391951,0.98050971,0.91639127,0.94723940
1,glm,0.91564403,0.88151416,0.89200029,0.88501516
2,gbm,0.97077894,0.97626046,0.95263089,0.96403432
3,deeplearn,0.93618280,0.98647110,0.88208695,0.93126931
4,cnn,0.52724034,0.13156732,0.16673663,0.14231620
5,bayes,0.90218935,0.93740541,0.82943131,0.87932768


,model,accuracy,precision,recall,f1_score
0,rf,0.00113408,-0.00321378,0.00189657,-0.00051347
1,glm,0.07158803,-0.02927362,0.15969995,0.07537674
2,gbm,0.03230318,-0.00197413,0.06375612,0.03277711
3,deeplearn,0.00070333,0.00302191,0.00155628,0.00223637
4,cnn,0.04232198,0.04145876,0.16595822,0.14079421
5,bayes,0.02721373,-0.00422575,0.05027182,0.02765012


accuracy    0.02921072
precision   0.00096556
recall      0.07385649
f1_score    0.04638684
dtype: float64

# 调试区

In [84]:
sample,df_baseline , df_best = analyse.FindBest("deeplearn","08")
display(sample, df_baseline, df_best )

,Cosine,Pearson,Euclidean,EMD,KS,Manhattan,Minkowski,Jaccard,Entropy,train,test
67,0.73910730,0.72611004,0.95155614,0.03553199,0.53747372,4.40601046,0.64690000,0.11712880,0.62505584,04,08


,model,train,test,acccuracy,precision,recall,f1_score
0,deeplearn,baseline,08,0.98051116,0.93302622,0.88683058,0.90934208


,model,train,test,acccuracy,precision,recall,f1_score
0,deeplearn,04,08,0.98426045,0.96834744,0.88615696,0.92543088


In [89]:
df_report['train'].unique()

array(['baseline', '02', '01', '04', '03'], dtype=object)

In [50]:
analyse.m_raw_similarity

,Cosine,Pearson,Euclidean,EMD,KS,Manhattan,Minkowski,Jaccard,Entropy,train,test
0,0.57112056,0.54905241,1.24033489,0.03839231,0.43988939,4.86836578,0.90560000,0.07296063,0.66847037,03,03
1,0.55265199,0.53809339,1.15435968,0.02614594,0.38074900,3.72603626,0.88280000,0.07978940,0.97808403,03,00
2,0.58892837,0.56822640,1.20381450,0.03661478,0.43603432,4.61562536,0.88390000,0.08702521,0.76565084,03,02
3,0.67419524,0.65610890,1.06710160,0.03856632,0.38026047,4.74454411,0.73650000,0.08464531,0.67927905,03,05
4,0.74154933,0.72784198,1.01890179,0.04101252,0.52734193,4.86788339,0.68460000,0.12004101,0.61719082,03,08
...,...,...,...,...,...,...,...,...,...,...,...
85,0.73928365,0.71235651,1.38462032,0.08455316,0.50070636,9.15257524,0.82180000,0.09759639,0.86018851,baseline,08
86,0.75363409,0.72595135,1.35939277,0.08434920,0.49155892,9.12597537,0.79880000,0.08669322,0.81519910,baseline,07
87,0.70037226,0.67351291,1.36956712,0.07855441,0.48263880,8.56682796,0.83970000,0.07457323,1.00211452,baseline,01
88,0.77372198,0.74714650,1.28791557,0.08011632,0.45381876,8.63026980,0.75340000,0.08436476,0.85964332,baseline,04
